# 🚦 Vietnamese Traffic Intelligence System (Colab GPU Training)
**Mục tiêu:** Huấn luyện siêu tốc mô hình YOLO11n Biển báo & Đèn giao thông trên GPU T4 miễn phí (50 Epochs trong ~10-12 phút).

### 1️⃣ Bước 1: Kiểm tra GPU Tesla T4

In [ ]:
!nvidia-smi
!pip install -q ultralytics

--- 
### 2️⃣ Bước 2: Nối Google Drive để lấy Data
* Kéo file `Processed_Dataset_For_Colab.zip` thả lên Google Drive của bạn.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Giải nén dữ liệu vào ổ cứng siêu tốc của Colab
!mkdir -p /content/dataset
!unzip -q /content/drive/MyDrive/Processed_Dataset_For_Colab.zip -d /content/dataset
print('Data extraction completed! Checking folders:')
!ls /content/dataset

--- 
### 3️⃣ Bước 3: Tạo file cấu hình data.yaml trên Colab

In [ ]:
yaml_signs = '''
path: /content/dataset/traffic_signs_yolo
train: train/images
val: val/images
test: test/images

nc: 7
names:
  0: Cam_nguoc_chieu
  1: Cam_dung_va_do
  2: Cam_re
  3: Gioi_han_toc_do
  4: Cam_con_lai
  5: Nguy_hiem
  6: Hieu_lenh
'''
with open('/content/dataset/signs_colab.yaml', 'w') as f:
    f.write(yaml_signs.strip())

yaml_lights = '''
path: /content/dataset/traffic_lights_yolo
train: train/images
val: val/images
test: test/images

nc: 2
names:
  0: Green
  1: Red
'''
with open('/content/dataset/lights_colab.yaml', 'w') as f:
    f.write(yaml_lights.strip())
print('Colab YAML configs ready!')

--- 
### 4️⃣ Bước 4: Huấn luyện Full 50 Epochs YOLO11n Biển Báo (GPU)

In [ ]:
from ultralytics import YOLO

model_signs = YOLO('yolo11n.pt')
results_signs = model_signs.train(
    data='/content/dataset/signs_colab.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    project='/content/runs/detect',
    name='traffic_signs_yolo11n_gpu',
    exist_ok=True
)

--- 
### 5️⃣ Bước 5: Huấn luyện Full 50 Epochs YOLO11n Đèn Giao Thông (GPU)

In [ ]:
model_lights = YOLO('yolo11n.pt')
results_lights = model_lights.train(
    data='/content/dataset/lights_colab.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    project='/content/runs/detect',
    name='traffic_lights_yolo11n_gpu',
    exist_ok=True
)

--- 
### 6️⃣ Bước 6: Tự động Đóng Gói Trọng Số & Biểu Đồ Tải Về Máy

In [ ]:
# Nén kết quả lại và tải về máy
!zip -r /content/trained_models_results.zip /content/runs/detect

from google.colab import files
files.download('/content/trained_models_results.zip')